# 🚲 Bike Demand Forecasting
## Step 2 & 3 — Data Intake & Data Quality Report

**Dataset:** Capital Bikeshare, Washington D.C. (2011–2012)  
**Source:** UCI Machine Learning Repository  
This notebook loads the dataset, builds a reusable pipeline, generates a data dictionary, and produces a full data quality report.

### 2.1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### 2.2 — Load Dataset
We load both the **daily** and **hourly** CSV files.

In [ ]:
# Reusable loader function
def load_data(day_path="day.csv", hour_path="hour.csv"):
    day_df  = pd.read_csv(day_path)
    hour_df = pd.read_csv(hour_path)
    return day_df, hour_df

day_df, hour_df = load_data()

print(f"day.csv  → {day_df.shape[0]:,} rows × {day_df.shape[1]} columns")
print(f"hour.csv → {hour_df.shape[0]:,} rows × {hour_df.shape[1]} columns")

### 2.3 — Data Dictionary
A structured description of every column.

In [ ]:
descriptions = {
    'instant'   : 'Record index',
    'dteday'    : 'Date (YYYY-MM-DD)',
    'season'    : 'Season — 1:Spring 2:Summer 3:Fall 4:Winter',
    'yr'        : 'Year — 0:2011  1:2012',
    'mnth'      : 'Month (1–12)',
    'hr'        : 'Hour (0–23) — hourly dataset only',
    'holiday'   : 'Public holiday flag (0/1)',
    'weekday'   : 'Day of week (0=Sun … 6=Sat)',
    'workingday': 'Working day flag (0/1)',
    'weathersit': 'Weather situation (1=Clear … 4=Heavy rain)',
    'temp'      : 'Normalized temperature in °C  [÷ 41]',
    'atemp'     : 'Normalized "feels-like" temperature  [÷ 50]',
    'hum'       : 'Normalized humidity  [÷ 100]',
    'windspeed' : 'Normalized wind speed  [÷ 67]',
    'casual'    : 'Casual (non-registered) user count',
    'registered': 'Registered user count',
    'cnt'       : 'Total rental count (casual + registered) ← TARGET',
}

data_dict = pd.DataFrame({
    'Column'     : list(descriptions.keys()),
    'Dtype'      : [str(hour_df[c].dtype) if c in hour_df.columns else 'N/A'
                    for c in descriptions],
    'Description': list(descriptions.values()),
})
data_dict

---
## Step 3 — Data Quality Report

### 3.1 — Row Counts (Before Cleaning)

In [ ]:
print("=== Before cleaning ===")
print(f"day.csv  rows : {len(day_df):,}")
print(f"hour.csv rows : {len(hour_df):,}")

### 3.2 — Missing Values

In [ ]:
print("--- Missing values in day.csv ---")
miss_day = day_df.isnull().sum()
print(miss_day[miss_day > 0] if miss_day.sum() else "✅ No missing values")

print("\n--- Missing values in hour.csv ---")
miss_hour = hour_df.isnull().sum()
print(miss_hour[miss_hour > 0] if miss_hour.sum() else "✅ No missing values")

### 3.3 — Duplicate Rows

In [ ]:
print(f"Duplicates in day.csv  : {day_df.duplicated().sum()}")
print(f"Duplicates in hour.csv : {hour_df.duplicated().sum()}")

### 3.4 — Outlier Detection (IQR method on `cnt`)

In [ ]:
def detect_outliers(series, label):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    print(f"{label}: Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}"
          f"  → {len(outliers)} outliers  (lower={lower:.0f}, upper={upper:.0f})")
    return outliers

detect_outliers(day_df['cnt'],  "day.csv  cnt")
detect_outliers(hour_df['cnt'], "hour.csv cnt")

### 3.5 — Summary Statistics (Continuous Variables)

In [ ]:
cols = ['temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
hour_df[cols].describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

### 3.6 — Row Counts (After Cleaning)

In [ ]:
# No rows removed — the dataset is already clean
print("=== After cleaning ===")
print(f"day.csv  rows : {len(day_df):,}  (unchanged)")
print(f"hour.csv rows : {len(hour_df):,}  (unchanged)")
print("\n✅ Dataset is clean — no imputation or row removal required.")